# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
pd.set_option('display.max_columns', None)

## Lottery and Sales Data (for reference when merging)

In [3]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [4]:
locations_df.head(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid
0,97637,TURCA GROCERY CORP,1142 E 165TH ST,BRONX,10459,-73.88748,40.82523,b'\x01\x01\x00\x00\x00\xa9\xf6\xe9x\xccxR\xc0\...,360050119005,36005011900
1,102843,VALENCIA BAKERY,499 E 138TH ST,BRONX,10454,-73.91962,40.80773,b'\x01\x01\x00\x00\x00\xd7/\xd8\r\xdbzR\xc0\xc...,360050039005,36005003900


In [5]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [6]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
12046,76589,120207.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,363.0,365,365.0,0.0,0.0,329.334247,329.334247,False,329.334247
12072,76758,436192.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,363.0,365,365.0,0.0,0.0,1195.047945,1195.047945,False,1195.047945
12795,82633,2756626.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,7552.400000,7552.400000,False,7552.400000
11712,74834,152586.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,363.0,365,365.0,0.0,0.0,418.043836,418.043836,False,418.043836
3058,109576,513222.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,1406.089041,1406.089041,False,1406.089041


## Overture Maps Data

In [7]:
overture_file_path = Path(".").resolve().parent / "data/interim/overture_filtered.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [10]:
overture_df.sample(5)

,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen
75,f5266768-d838-4444-b13a-ca1c0ff7b5dd,b'\x01\x01\x00\x00\x00\xa5\x9d?\x12\x1b;R\xc0\...,-72.923527,40.888523,food_and_beverage_store,grocery_store,"[shopping, food_and_beverage_store, grocery_st...",0.959614,NaN,2025-06-25.0
40,3d4f7f37-9e2c-4977-8b81-7decc2ae08a5,b'\x01\x01\x00\x00\x00\xfc\xaa\\\xa8\xfc=R\xc0...,-72.968546,40.894542,shopping,shopping,[shopping],0.850000,NaN,2025-06-25.0
66,d58a3113-0b7a-417a-ab5a-bff34c15a72d,b'\x01\x01\x00\x00\x00\x8bS\xad\x85Y<R\xc03\x8...,-72.942964,40.893649,gym,gym,"[sports_and_recreation, sport_or_fitness_facil...",0.997848,NaN,2025-06-25.0
79,3ef5c24b-7c43-49d2-b7b3-091be46d5443,"b""\x01\x01\x00\x00\x00|P'\x0b%;R\xc0\x04\xad\x...",-72.924136,40.887930,gym,gym,"[sports_and_recreation, sport_or_fitness_facil...",0.972216,Gold's Gym,2025-06-25.0
93,bc44d27b-3dc1-41b6-833f-281159a592f7,"b""\x01\x01\x00\x00\x00\x81i\x07\x12\xca9R\xc0\...",-72.902958,40.891392,gas_station,gas_station,"[travel_and_transportation, fueling_station, g...",0.999980,Exxon & Mobil Stations,2025-06-25.0
